In [9]:
import IPython.core.pylabtools
import IPython.core.pylabtools
import os
import sys
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import mlflow
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import time
import itertools
from joblib import Parallel, delayed

# Ask TensorFlow to list all available physical GPUs
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print(f"✅ M3 Pro GPU ACTIVATED! Found: {gpu_devices}")
    # Optional: Set memory growth to prevent TF from hoarding all unified memory
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print("❌ GPU not found. TensorFlow is falling back to the CPU.")

✅ M3 Pro GPU ACTIVATED! Found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data

In [1]:
import yfinance as yf
import pandas as pd
import warnings

# Désactiver les avertissements de yfinance pour garder une console propre
warnings.filterwarnings('ignore')

def download_weekly_commodities():
    # 🌍 L'Univers de base des Matières Premières (Les plus liquides)
    commodities_dict = {
# --- LES BOUCLIERS DÉFENSIFS (Safe Havens) ---
        "GC=F": "Gold",                 # Le roi des métaux précieux (Défensif)
        "ZN=F": "10-Year T-Note",       # Obligation d'État US à 10 ans (Bouclier anti-krach)
        "ZB=F": "30-Year T-Bond",
        "XLE": "Energy"       # Obligation d'État US à 30 ans (Ultra-réactif aux taux)
    }
    
    tickers = list(commodities_dict.keys())
    
    print(f"🚀 Lancement du téléchargement pour {len(tickers)} matières premières (Weekly)...")
    
    all_data = []
    
    # On boucle sur les tickers pour forcer un format 'Long' (Base de données) très propre
    for ticker in tickers:
        try:
            # L'astuce est ici : interval="1wk"
            df_ticker = yf.download(ticker, start="2000-01-01", interval="1wk", progress=False)
            
            if not df_ticker.empty:
                df_ticker = df_ticker.reset_index()
                
                # Yahoo renvoie parfois 'Date' ou 'Datetime', on normalise en minuscule
                date_col = 'Date' if 'Date' in df_ticker.columns else 'Datetime'
                
                # On ne garde que les colonnes utiles
                df_clean = df_ticker[[date_col, 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
                df_clean.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
                
                # On ajoute nos identifiants
                df_clean['symbol'] = ticker
                df_clean['name'] = commodities_dict[ticker]
                
                # Nettoyage des NaN qui peuvent survenir sur des semaines fériées
                df_clean = df_clean.dropna(subset=['close'])
                
                all_data.append(df_clean)
                print(f"✅ {ticker} ({commodities_dict[ticker]}) : {len(df_clean)} semaines récupérées.")
            else:
                print(f"⚠️ Aucune donnée pour {ticker}.")
                
        except Exception as e:
            print(f"❌ Erreur sur {ticker} : {e}")

    # Fusion de tous les DataFrames
    print("\n🧩 Fusion de la base de données...")
    df_final = pd.concat(all_data, ignore_index=True)
    
    # On trie chronologiquement et par actif
    df_final['date'] = pd.to_datetime(df_final['date']).dt.date
    df_final = df_final.sort_values(['symbol', 'date']).reset_index(drop=True)
    
    print(f"🎉 Terminé ! Mega-DataFrame créé avec {len(df_final)} semaines de cotation.")
    return df_final

# Exécution du script
if __name__ == "__main__":
    df_commodities_weekly = download_weekly_commodities()
    
    # Affichage d'un échantillon pour vérifier
    display(df_commodities_weekly.head())
    
    # Optionnel : Sauvegarde en local pour éviter de retélécharger
    # df_commodities_weekly.to_parquet("commodities_weekly_2000.parquet", index=False)

🚀 Lancement du téléchargement pour 4 matières premières (Weekly)...
✅ GC=F (Gold) : 1335 semaines récupérées.
✅ ZN=F (10-Year T-Note) : 1331 semaines récupérées.
✅ ZB=F (30-Year T-Bond) : 1332 semaines récupérées.
✅ XLE (Energy) : 1369 semaines récupérées.

🧩 Fusion de la base de données...
🎉 Terminé ! Mega-DataFrame créé avec 5367 semaines de cotation.


,date,open,high,low,close,volume,symbol,name
0,2000-08-28,273.899994,278.299988,273.899994,277.000000,0,GC=F,Gold
1,2000-09-04,275.799988,275.799988,273.299988,273.299988,127,GC=F,Gold
2,2000-09-11,273.100006,273.100006,272.299988,272.299988,0,GC=F,Gold
3,2000-09-18,271.399994,271.899994,269.000000,271.799988,0,GC=F,Gold
4,2000-09-25,274.100006,278.399994,272.000000,273.600006,4164,GC=F,Gold


# Backtest

In [13]:
import numpy as np
import pandas as pd
import itertools
import time
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings('ignore')

def generate_dual_momentum_signals(df, momentum_window, ma_fast_window, ma_slow_window, top_n, rebalance_freq='M'):
    """
    Stratégie Dual Momentum : Filtre de Tendance + Sélection des Top N selon un momentum spécifique.
    """
    data = df[['symbol', 'date', 'close']].copy()
    
    # 🛡️ Sécurité absolue sur le format des dates
    data['date'] = pd.to_datetime(data['date'])
    data = data.sort_values(['symbol', 'date'])

    # --- 1. ABSOLUTE MOMENTUM (Filtre de Tendance) ---
    data['ma_fast'] = data.groupby('symbol')['close'].transform(lambda x: x.rolling(ma_fast_window).mean())
    data['ma_slow'] = data.groupby('symbol')['close'].transform(lambda x: x.rolling(ma_slow_window).mean())
    
    # L'actif n'est éligible à l'achat QUE si sa tendance est positive
    data['Is_Uptrend'] = data['ma_fast'] > data['ma_slow']

    # --- 2. RELATIVE MOMENTUM (Score pour le classement) ---
    # 👈 NOUVEAUTÉ : On utilise la variable indépendante momentum_window
    data['Momentum_Score'] = data.groupby('symbol')['close'].pct_change(periods=momentum_window)

    # Rendement futur
    data['NextReturn'] = data.groupby('symbol')['close'].shift(-1) / data['close'] - 1

    # --- 3. GESTION DU REBALANCEMENT ---
    if rebalance_freq == 'W':
        data['is_rebalance_date'] = True
    elif rebalance_freq == 'M':
        data['Period'] = data['date'].dt.to_period('M')
        rebalance_dates = data.groupby('Period')['date'].transform('max')
        data['is_rebalance_date'] = (data['date'] == rebalance_dates)
    elif rebalance_freq == 'Q':
        data['Period'] = data['date'].dt.to_period('Q')
        rebalance_dates = data.groupby('Period')['date'].transform('max')
        data['is_rebalance_date'] = (data['date'] == rebalance_dates)
    elif rebalance_freq == 'Y':
        data['Period'] = data['date'].dt.to_period('Y')
        rebalance_dates = data.groupby('Period')['date'].transform('max')
        data['is_rebalance_date'] = (data['date'] == rebalance_dates)

    # --- 4. SÉLECTION DU TOP N ---
    reb_data = data[data['is_rebalance_date']].copy()
    
    # On ne garde que les actifs qui sont en tendance haussière
    eligible = reb_data[reb_data['Is_Uptrend'] == True].copy()
    
    # On classe ces actifs éligibles selon leur score de Momentum
    eligible['Rank'] = eligible.groupby('date')['Momentum_Score'].rank(method='first', ascending=False)
    
    # On sélectionne les N meilleurs
    buys = eligible[eligible['Rank'] <= top_n].copy()
    
    # Équi-pondération sur les actifs sélectionnés
    buys['Target_Weight'] = 1.0 / top_n

    # --- 5. RÉINTÉGRATION ET PROPAGATION ---
    data = data.merge(buys[['symbol', 'date', 'Target_Weight']], on=['symbol', 'date'], how='left')
    
    # Les jours de rebalancement où l'actif n'est pas sélectionné, son poids tombe à 0
    data.loc[data['is_rebalance_date'] & data['Target_Weight'].isna(), 'Target_Weight'] = 0
    
    # L'action reste en portefeuille jusqu'au prochain rebalancement (Forward Fill)
    data['Target_Weight'] = data.groupby('symbol')['Target_Weight'].ffill().fillna(0)

    # Nettoyage
    return data.dropna(subset=['NextReturn', 'ma_slow', 'Momentum_Score'])

def run_vectorized_backtest(data, transaction_cost=0.001):
    data['Strat_Return'] = data['Target_Weight'] * data['NextReturn']
    
    # Frais de transaction
    data['Weight_Change'] = data.groupby('symbol')['Target_Weight'].diff().fillna(data['Target_Weight'])
    data['Cost'] = data['Weight_Change'].abs() * transaction_cost
    
    port_returns = data.groupby('date')[['Strat_Return', 'Cost']].sum()
    port_returns['Net_Return'] = port_returns['Strat_Return'] - port_returns['Cost']
    
    # Attention au Cash ! Si on demande le Top 3 mais qu'un seul actif monte, 
    # le poids total sera de 33%. Les 67% restants sont virtuellement en Cash (rendement 0).
    port_returns['Capital'] = (1 + port_returns['Net_Return']).cumprod()
    
    return port_returns

def run_single_backtest(params, df_source, start_date, end_date):
    mom_win, ma_fast, ma_slow, top_n, reb_freq = params

    default_output = {
        "Mom_Window": mom_win, "MA_Fast": ma_fast, "MA_Slow": ma_slow, 
        "Top_N": top_n, "Rebalance": reb_freq,
        "Total Return": np.nan, "CAGR": np.nan, "Max Drawdown": np.nan,
        "Sharpe Ratio": np.nan, "Error": None
    }

    try:
        if ma_fast >= ma_slow:
            default_output["Error"] = "Invalid MA Config"
            return default_output

        full_signals = generate_dual_momentum_signals(
            df_source, momentum_window=mom_win, ma_fast_window=ma_fast, ma_slow_window=ma_slow, 
            top_n=top_n, rebalance_freq=reb_freq
        )

        mask = (full_signals['date'] >= pd.to_datetime(start_date)) & \
               (full_signals['date'] <= pd.to_datetime(end_date))
        backtest_data = full_signals.loc[mask]

        if backtest_data.empty:
            default_output["Error"] = "No data in timeframe"
            return default_output

        res_df = run_vectorized_backtest(backtest_data, transaction_cost=0.001)

        total_return = res_df['Capital'].iloc[-1] - 1
        # Divisé par 365.25 pour un calcul précis des années
        n_years = (res_df.index[-1] - res_df.index[0]).days / 365.25
        cagr = (res_df['Capital'].iloc[-1]) ** (1 / max(1, n_years)) - 1
        
        rolling_max = res_df['Capital'].cummax()
        max_dd = ((res_df['Capital'] - rolling_max) / rolling_max).min()
        
        mean_ret = res_df['Net_Return'].mean()
        std_ret = res_df['Net_Return'].std()
        sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret > 0 else 0 

        output = default_output.copy()
        output.update({
            "Total Return": total_return, "CAGR": cagr, 
            "Max Drawdown": max_dd, "Sharpe Ratio": sharpe
        })
        return output

    except Exception as e:
        default_output["Error"] = str(e)
        return default_output

def grid_search_execution(df, param_grid, start_date, end_date):
    keys, values = zip(*param_grid.items())
    combinations = [v for v in itertools.product(*values)]

    print(f"🚀 Lancement de la Grid Search sur {len(combinations)} combinaisons...")
    start_time = time.time()

    results_list = Parallel(n_jobs=-1)(
        delayed(run_single_backtest)(params, df, start_date, end_date) for params in combinations
    )

    end_time = time.time()
    print(f"✅ Terminé en {end_time - start_time:.2f} secondes.")

    results_df = pd.DataFrame(results_list)
    
    # Affichage sécurisé des erreurs
    erreurs = results_df[results_df['Error'].notna()]
    if not erreurs.empty:
        print(f"⚠️ {len(erreurs)} combinaisons ont été ignorées (Configs invalides ou erreurs).")

    best_strats = results_df[results_df['Error'].isna()].sort_values(by='Sharpe Ratio', ascending=False)
    return best_strats

# ==========================================
# 🚀 EXÉCUTION
# ==========================================
param_grid = {
    'momentum_window': [12, 26, 52], # 👈 NOUVEAU : Score basé sur 3 mois, 6 mois ou 1 an
    'ma_fast': [12, 26],             # 3 mois, 6 mois
    'ma_slow': [26, 52],             # 6 mois, 1 an
    'top_n': [1, 2, 3, 4],           # Sélection du Top N
    'rebalance_freq': ['W', 'M', 'Q', 'Y']  # Rebalancement
}

# ⚠️ N'oublie pas d'utiliser le dataframe approprié (df_global_macro est idéal ici)
best_strategies_df = grid_search_execution(
    df=df_commodities_weekly, 
    param_grid=param_grid,
    start_date='2008-01-01',
    end_date='2010-03-01'
)

print("\n🏆 Top 10 des configurations Dual Momentum (Sérénité) :")
display(best_strategies_df.head(10))

🚀 Lancement de la Grid Search sur 192 combinaisons...


/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652/1565894526.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652/1565894526.py:81: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652/1565894526.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the doc

✅ Terminé en 0.42 secondes.
⚠️ 48 combinaisons ont été ignorées (Configs invalides ou erreurs).

🏆 Top 10 des configurations Dual Momentum (Sérénité) :


y using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652/1565894526.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652/1565894526.py:81: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/6k/82j2nnl13hj9ld7nmzpdt6fh0000gn/T/ipykernel_20652

,Mom_Window,MA_Fast,MA_Slow,Top_N,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
133,52,12,26,2,M,0.197220,0.087244,-0.160036,0.425657,None
69,26,12,26,2,M,0.192650,0.085313,-0.159212,0.414316,None
65,26,12,26,1,M,0.276488,0.120122,-0.334448,0.383074,None
64,26,12,26,1,W,0.252400,0.110250,-0.297993,0.365483,None
5,12,12,26,2,M,0.160156,0.071471,-0.182152,0.363334,None
129,52,12,26,1,M,0.239133,0.104769,-0.353925,0.350208,None
68,26,12,26,2,W,0.151442,0.067724,-0.177123,0.347927,None
132,52,12,26,2,W,0.145989,0.065371,-0.177123,0.343762,None
73,26,12,26,3,M,0.108357,0.048969,-0.111155,0.338897,None
9,12,12,26,3,M,0.108357,0.048969,-0.111155,0.338897,None


## Running only one combination

In [14]:
best_strategies_df.sort_values('CAGR', ascending=False).head(20)

,Mom_Window,MA_Fast,MA_Slow,Top_N,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
65,26,12,26,1,M,0.276488,0.120122,-0.334448,0.383074,None
64,26,12,26,1,W,0.252400,0.110250,-0.297993,0.365483,None
129,52,12,26,1,M,0.239133,0.104769,-0.353925,0.350208,None
1,12,12,26,1,M,0.201238,0.088938,-0.356780,0.314934,None
133,52,12,26,2,M,0.197220,0.087244,-0.160036,0.425657,None
69,26,12,26,2,M,0.192650,0.085313,-0.159212,0.414316,None
128,52,12,26,1,W,0.188257,0.083454,-0.333948,0.308162,None
0,12,12,26,1,W,0.182962,0.081208,-0.357066,0.300099,None
5,12,12,26,2,M,0.160156,0.071471,-0.182152,0.363334,None
68,26,12,26,2,W,0.151442,0.067724,-0.177123,0.347927,None


In [15]:
best_strategies_df.sort_values('Sharpe Ratio', ascending=False).head(20)

,Mom_Window,MA_Fast,MA_Slow,Top_N,Rebalance,Total Return,CAGR,Max Drawdown,Sharpe Ratio,Error
133,52,12,26,2,M,0.197220,0.087244,-0.160036,0.425657,None
69,26,12,26,2,M,0.192650,0.085313,-0.159212,0.414316,None
65,26,12,26,1,M,0.276488,0.120122,-0.334448,0.383074,None
64,26,12,26,1,W,0.252400,0.110250,-0.297993,0.365483,None
5,12,12,26,2,M,0.160156,0.071471,-0.182152,0.363334,None
129,52,12,26,1,M,0.239133,0.104769,-0.353925,0.350208,None
68,26,12,26,2,W,0.151442,0.067724,-0.177123,0.347927,None
132,52,12,26,2,W,0.145989,0.065371,-0.177123,0.343762,None
73,26,12,26,3,M,0.108357,0.048969,-0.111155,0.338897,None
9,12,12,26,3,M,0.108357,0.048969,-0.111155,0.338897,None
